# 05 · Mjolnir limits – per-user token quotas
`llm-token-limit` enforces tokens-per-minute **per subscription**. `avengers-initiative` users (thor, loki, hulk, black-widow) get a small budget; `tony-stark` (`infinity-gauntlet`) a large one.
Exceeding it returns **429** – only for that user.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

# 40 calls x ~330 tokens = ~13000 tokens: far above hulk's 5000 tokens/min, well below tony-stark's 100000.
pd.DataFrame([burst(user, n=40, prompt="Write a 150-word story about Mjolnir.", max_completion_tokens=300)
              for user in ('hulk', 'tony-stark')])

### What did the gateway see? (last 5 min)
Every request incl. the 429s is in `ApiManagementGatewayLogs` (a few minutes of lag). Throttled requests never reach the model, so they are **not** in `ApiManagementGatewayLlmLog`.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

throttling_by_user(minutes=5)   # polls until the burst above shows up in the gateway log

### Allowed vs throttled per minute
The budget refills every minute, so bursts show up as bars of 429s.

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

throttling_per_minute(minutes=15)

### Throttled by the gateway or by the model?

In [ ]:
import sys; sys.path.insert(0, '.')
from heimdal_helpers import *

throttling_source(minutes=15)   # empty BackendResponseCode = gateway token limit, 429 = Foundry itself